# TechCorp — Fine-tuning médical LoRA (Phi-3-mini)

**Dataset :** `medical_dataset_clean.json` (1000 échantillons)

1. Runtime → T4 GPU
2. Uploader le dataset nettoyé depuis `rendu/data/`
3. Exécuter toutes les cellules

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

In [ ]:
from google.colab import files
uploaded = files.upload()  # Sélectionner medical_dataset_clean.json
DATASET_PATH = list(uploaded.keys())[0]
print(f"Dataset chargé : {DATASET_PATH}")

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
OUTPUT_DIR = "./phi3_medical_lora"

with open(DATASET_PATH, encoding="utf-8") as f:
    raw = json.load(f)

texts = []
for item in raw:
    instruction = item.get("instruction") or item.get("input", "")
    output = item.get("output") or item.get("answer", "")
    if instruction and output:
        text = f"<|user|>\n{instruction}<|end|>\n<|assistant|>\n{output}<|end|>"
        texts.append({"text": text})

print(f"{len(texts)} exemples préparés")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

hf_dataset = Dataset.from_list(texts)
tokenized_dataset = hf_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    save_steps=200,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

In [ ]:
final_loss = trainer.state.log_history[-1].get("loss", "N/A")
print(f"Final loss: {final_loss}")
print(f"Total steps: {trainer.state.global_step}")
print(f"Epochs: {training_args.num_train_epochs}")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Modèle sauvegardé dans {OUTPUT_DIR}")

In [ ]:
test_prompt = "I have chest pain when exercising. Should I be worried?"
formatted = f"<|user|>\n{test_prompt}<|end|>\n<|assistant|>\n"
inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))